# 02 – Model Training

Train a Logistic Regression fraud detection model, evaluate it comprehensively,
and save the artefacts for use by the streaming and batch components.

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score,
    f1_score,
)

%matplotlib inline
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Load Data

In [ ]:
CSV_PATH    = '../data/creditcard.csv'
MODEL_DIR   = '../models'
os.makedirs(MODEL_DIR, exist_ok=True)

if not os.path.exists(CSV_PATH):
    sys.path.insert(0, '../data')
    from generate_synthetic_data import generate_synthetic_data
    df = generate_synthetic_data(n_samples=100_000, fraud_ratio=0.02)
    df.to_csv(CSV_PATH, index=False)
else:
    df = pd.read_csv(CSV_PATH)

print(f'Dataset shape: {df.shape}')
print(f'Fraud rate:    {df["Class"].mean()*100:.2f}%')
df.head()

## 2. Preprocessing & Train/Test Split

In [ ]:
FEATURE_COLS = [f'V{i}' for i in range(1, 29)] + ['Amount', 'Time']
TARGET_COL   = 'Class'

X = df[FEATURE_COLS].values
y = df[TARGET_COL].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print(f'Train: {len(X_train):,} | Test: {len(X_test):,}')
print(f'Train fraud: {y_train.sum():,} | Test fraud: {y_test.sum():,}')

## 3. Train Logistic Regression

In [ ]:
model = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    random_state=42,
    solver='lbfgs',
)
model.fit(X_train, y_train)
print('Training complete.')

## 4. Evaluation

In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print('Classification Report')
print('=' * 50)
print(classification_report(y_test, y_pred, target_names=['Normal', 'Fraud']))
print(f'ROC-AUC : {roc_auc_score(y_test, y_prob):.4f}')
print(f'F1 Score: {f1_score(y_test, y_pred):.4f}')

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Normal', 'Fraud'],
            yticklabels=['Normal', 'Fraud'])
plt.title('Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

In [ ]:
# ROC curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
auc = roc_auc_score(y_test, y_prob)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {auc:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=1, linestyle='--', label='Random classifier')
plt.xlim([0, 1])
plt.ylim([0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# Precision-Recall curve
precision, recall, _ = precision_recall_curve(y_test, y_prob)
ap = average_precision_score(y_test, y_prob)

plt.figure(figsize=(8, 6))
plt.plot(recall, precision, color='green', lw=2, label=f'PR curve (AP = {ap:.4f})')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend(loc='upper right')
plt.tight_layout()
plt.show()

## 5. Save Model & Scaler

In [ ]:
model_path  = os.path.join(MODEL_DIR, 'fraud_model.pkl')
scaler_path = os.path.join(MODEL_DIR, 'scaler.pkl')

joblib.dump(model,  model_path)
joblib.dump(scaler, scaler_path)

print(f'Model  saved → {model_path}')
print(f'Scaler saved → {scaler_path}')

## 6. Test Model Loading & Inference

In [ ]:
loaded_model  = joblib.load(model_path)
loaded_scaler = joblib.load(scaler_path)

# Single transaction inference
rng = np.random.RandomState(0)
sample = rng.randn(1, len(FEATURE_COLS))
sample_scaled = loaded_scaler.transform(sample)
prediction    = loaded_model.predict(sample_scaled)[0]
probability   = loaded_model.predict_proba(sample_scaled)[0][1]

print(f'Sample prediction : {"FRAUD" if prediction == 1 else "NORMAL"}')
print(f'Fraud probability : {probability:.4f}')